# One-vs-Rest Classification Error Analysis

## Brain Connectivity Classification: Decomposed Binary Classification Approach

---

### Research Context

This notebook implements One-vs-Rest (OvR) classification for brain region identification, building on limitations identified in the multinomial analysis. OvR decomposes the 232-class problem into 232 independent binary tasks.

### Hypotheses

| Hypothesis | Multinomial Baseline | OvR Prediction |
|------------|---------------------|----------------|
| H1: Accuracy | 92.4% CV, 89.2% Task | Higher due to specialized classifiers |
| H2: Confidence | 7-9% correct <50% conf | Better calibrated binary probabilities |
| H3: Error patterns | 88% cross boundaries | More within-network errors |
| H4: Task sensitivity | Uniform degradation | Differential sensitivity |

### Notebook Structure

1. Setup & Data Loading
2. OvR Model Training
3. Performance Overview
4. Comparison with Multinomial
5. Error Taxonomy Analysis
6. Confidence Calibration
7. Network-Level Patterns
8. Per-Region Analysis
9. Task-Induced Reorganization
10. Conclusions

---
## 1. Setup & Data Loading

In [2]:
# Core Libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings
import pickle
from collections import defaultdict

# Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, brier_score_loss,
                             balanced_accuracy_score)
from sklearn.calibration import calibration_curve

# Statistical Analysis
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Progress tracking
from tqdm.notebook import tqdm

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

print("✓ Libraries imported")

✓ Libraries imported


In [3]:
# Path Configuration
PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = DATA_DIR / 'results'
OVR_DIR = RESULTS_DIR / 'ovr_analysis'

# Create output directories
OVR_DIR.mkdir(parents=True, exist_ok=True)
(OVR_DIR / 'full').mkdir(exist_ok=True)
(OVR_DIR / 'left').mkdir(exist_ok=True)
(OVR_DIR / 'right').mkdir(exist_ok=True)

# Multinomial results paths
MULTI_DIR = RESULTS_DIR / 'full_connectivity_analysis' / 'multinomial'
TASK_DIR = RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing'

print(f"Results Directory: {OVR_DIR}")

Results Directory: /home/sjoon/projects/brain_connectivity_classifier/data/results/ovr_analysis


In [4]:
# Helper Functions
def load_json(fp): 
    with open(fp, 'r') as f: return json.load(f)
def save_json(data, fp): 
    with open(fp, 'w') as f: json.dump(data, f, indent=2)
def load_npy(fp): 
    return np.load(fp, allow_pickle=True)
def load_csv(fp): 
    return pd.read_csv(fp)

print("✓ Helper functions defined")

✓ Helper functions defined


In [ ]:
# Load connectivity data
# Modify paths according to your data structure

X_rest = load_npy(DATA_DIR / 'processed' / 'rest_connectivity.npy')
y_rest = load_npy(DATA_DIR / 'processed' / 'rest_labels.npy')
X_task = load_npy(DATA_DIR / 'processed' / 'task_connectivity.npy')
y_task = load_npy(DATA_DIR / 'processed' / 'task_labels.npy')

# Load region info
region_info = load_csv(MULTI_DIR / 'region_info.csv')

print(f"Rest: X={X_rest.shape}, y={y_rest.shape}")
print(f"Task: X={X_task.shape}, y={y_task.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/sjoon/projects/brain_connectivity_classifier/data/rest_connectivity.npy'

In [ ]:
# Load multinomial results for comparison
multi_metrics = load_json(MULTI_DIR / 'overall_metrics.json')
multi_task = load_json(TASK_DIR / 'task_testing_summary.json')
multi_cv_preds = load_npy(MULTI_DIR / 'cv_predictions.npy')
multi_cv_probs = load_npy(MULTI_DIR / 'cv_probabilities.npy')
multi_cv_true = load_npy(MULTI_DIR / 'cv_true_labels.npy')
multi_task_preds = load_npy(TASK_DIR / 'task_predictions.npy')
multi_task_probs = load_npy(TASK_DIR / 'task_probabilities.npy')
multi_task_true = load_npy(TASK_DIR / 'task_true_labels.npy')
multi_cm_cv = load_npy(MULTI_DIR / 'confusion_matrix.npy')
multi_cm_task = load_npy(TASK_DIR / 'task_confusion_matrix.npy')

print(f"Multinomial CV Acc: {multi_metrics['accuracy']:.4f}")
print(f"Multinomial Task Acc: {multi_task['task_test_accuracy']:.4f}")

In [ ]:
# Network mapping
network_to_major = {
    'VisCent': 'Visual', 'VisPeri': 'Visual',
    'SomMotA': 'Somatomotor', 'SomMotB': 'Somatomotor',
    'DorsAttnA': 'Dorsal Attention', 'DorsAttnB': 'Dorsal Attention',
    'SalVentAttnA': 'Salience/Ventral Attention', 'SalVentAttnB': 'Salience/Ventral Attention',
    'LimbicA': 'Limbic', 'LimbicB': 'Limbic',
    'ContA': 'Control', 'ContB': 'Control', 'ContC': 'Control',
    'DefaultA': 'Default', 'DefaultB': 'Default', 'DefaultC': 'Default',
    'TempPar': 'Default',
    'Hippocampus_ant': 'Subcortical', 'Hippocampus_post': 'Subcortical',
    'Amygdala_lat': 'Subcortical', 'Amygdala_med': 'Subcortical',
    'Thalamus_DA': 'Subcortical', 'Thalamus_DP': 'Subcortical',
    'Thalamus_VA': 'Subcortical', 'Thalamus_VP': 'Subcortical',
    'Caudate_ant': 'Subcortical', 'Caudate_post': 'Subcortical',
    'Putamen_ant': 'Subcortical', 'Putamen_post': 'Subcortical',
    'Pallidum_ant': 'Subcortical', 'Pallidum_post': 'Subcortical',
    'Accumbens_core': 'Subcortical', 'Accumbens_shell': 'Subcortical'
}

region_info['major_network'] = region_info['network'].map(network_to_major)

NETWORKS = ['Visual', 'Somatomotor', 'Dorsal Attention', 'Salience/Ventral Attention',
            'Limbic', 'Control', 'Default', 'Subcortical']
N_REGIONS = 232
N_FOLDS = 5

print("✓ Constants defined")

---
## 2. OvR Model Training

Train N binary classifiers, each distinguishing one region from all others.

In [ ]:
class OvRClassifier:
    """One-vs-Rest classifier for brain region identification."""
    
    def __init__(self, n_regions, region_info, random_state=42):
        self.n_regions = n_regions
        self.region_info = region_info
        self.random_state = random_state
        self.classifiers = {}
        self.cv_results = {}
        self.is_fitted = False
    
    def fit_with_cv(self, X, y, n_folds=5, verbose=True):
        """Train all binary classifiers with cross-validation."""
        self.n_samples = len(y)
        self.cv_probs_matrix = np.zeros((self.n_samples, self.n_regions))
        
        cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=self.random_state)
        iterator = tqdm(range(self.n_regions), desc="Training") if verbose else range(self.n_regions)
        
        for region_id in iterator:
            y_binary = (y == region_id).astype(int)
            
            clf = LogisticRegression(
                class_weight='balanced', penalty='l2', solver='lbfgs',
                max_iter=1000, random_state=self.random_state, n_jobs=-1
            )
            
            cv_probs = cross_val_predict(clf, X, y_binary, cv=cv, method='predict_proba')
            cv_preds = cross_val_predict(clf, X, y_binary, cv=cv)
            
            self.cv_probs_matrix[:, region_id] = cv_probs[:, 1]
            
            clf.fit(X, y_binary)
            
            self.cv_results[region_id] = {
                'accuracy': accuracy_score(y_binary, cv_preds),
                'balanced_acc': balanced_accuracy_score(y_binary, cv_preds),
                'precision': precision_score(y_binary, cv_preds, zero_division=0),
                'recall': recall_score(y_binary, cv_preds, zero_division=0),
                'f1': f1_score(y_binary, cv_preds, zero_division=0),
                'auc': roc_auc_score(y_binary, cv_probs[:, 1]) if y_binary.sum() > 0 else 0.5,
                'brier': brier_score_loss(y_binary, cv_probs[:, 1])
            }
            
            self.classifiers[region_id] = clf
        
        self.cv_predictions = np.argmax(self.cv_probs_matrix, axis=1)
        self.cv_true_labels = y
        self.is_fitted = True
        
        if verbose:
            acc = accuracy_score(y, self.cv_predictions)
            print(f"\n✓ CV Accuracy: {acc:.4f}")
    
    def predict(self, X):
        """Predict using argmax of binary probabilities."""
        prob_matrix = np.column_stack([
            self.classifiers[r].predict_proba(X)[:, 1] for r in range(self.n_regions)
        ])
        return np.argmax(prob_matrix, axis=1), prob_matrix
    
    def get_results_df(self):
        """Return CV results as DataFrame."""
        df = pd.DataFrame(self.cv_results).T.reset_index()
        df.columns = ['region_id'] + list(df.columns[1:])
        df = df.merge(self.region_info[['region_idx', 'region_name', 'major_network', 'hemisphere']],
                      left_on='region_id', right_on='region_idx', how='left')
        return df
    
    def save(self, path):
        with open(path, 'wb') as f: pickle.dump(self, f)
    
    @classmethod
    def load(cls, path):
        with open(path, 'rb') as f: return pickle.load(f)

print("✓ OvRClassifier defined")

In [ ]:
# Train Full Model
print("="*60)
print("TRAINING FULL MODEL (232 regions)")
print("="*60)

ovr = OvRClassifier(n_regions=N_REGIONS, region_info=region_info, random_state=42)
ovr.fit_with_cv(X_rest, y_rest, n_folds=N_FOLDS, verbose=True)
ovr.save(OVR_DIR / 'full' / 'ovr_model.pkl')

In [ ]:
# Test on Task Data
print("\nTesting on task data...")
task_preds, task_probs = ovr.predict(X_task)
task_acc = accuracy_score(y_task, task_preds)
print(f"Task Accuracy: {task_acc:.4f}")

# Save predictions
np.save(OVR_DIR / 'full' / 'cv_predictions.npy', ovr.cv_predictions)
np.save(OVR_DIR / 'full' / 'cv_probabilities.npy', ovr.cv_probs_matrix)
np.save(OVR_DIR / 'full' / 'task_predictions.npy', task_preds)
np.save(OVR_DIR / 'full' / 'task_probabilities.npy', task_probs)

---
## 3. Performance Overview

In [ ]:
# Aggregate metrics
ovr_cv_acc = accuracy_score(ovr.cv_true_labels, ovr.cv_predictions)
ovr_task_acc = task_acc

print("="*80)
print("PERFORMANCE SUMMARY")
print("="*80)
print(f"\n{'Metric':<20} {'OvR':>15} {'Multinomial':>15} {'Difference':>15}")
print("-"*65)
print(f"{'CV Accuracy':<20} {ovr_cv_acc:>14.2%} {multi_metrics['accuracy']:>14.2%} "
      f"{(ovr_cv_acc - multi_metrics['accuracy'])*100:>+14.2f}%")
print(f"{'Task Accuracy':<20} {ovr_task_acc:>14.2%} {multi_task['task_test_accuracy']:>14.2%} "
      f"{(ovr_task_acc - multi_task['task_test_accuracy'])*100:>+14.2f}%")
print(f"{'Accuracy Drop':<20} {(ovr_cv_acc - ovr_task_acc):>14.2%} "
      f"{(multi_metrics['accuracy'] - multi_task['task_test_accuracy']):>14.2%}")
print("="*80)

In [ ]:
# Per-classifier metrics
cv_results_df = ovr.get_results_df()

print("\nPER-CLASSIFIER METRICS SUMMARY:")
print(cv_results_df[['auc', 'f1', 'precision', 'recall']].describe().round(4))

print("\n" + "-"*60)
print("TOP 10 CLASSIFIERS (by AUC):")
print(cv_results_df.nlargest(10, 'auc')[['region_name', 'major_network', 'auc', 'f1']].to_string(index=False))

print("\nBOTTOM 10 CLASSIFIERS (by AUC):")
print(cv_results_df.nsmallest(10, 'auc')[['region_name', 'major_network', 'auc', 'f1']].to_string(index=False))

In [ ]:
# AUC distribution by network
fig = px.box(cv_results_df, x='major_network', y='auc', color='major_network',
             points='all', hover_data=['region_name'])
fig.update_layout(
    title='<b>Per-Classifier AUC by Network</b>',
    xaxis_title='Network', yaxis_title='AUC-ROC',
    template='simple_white', height=500, showlegend=False
)
fig.update_xaxes(tickangle=45)
fig.show()

---
## 4. Comparison with Multinomial

In [ ]:
# McNemar's Test
def mcnemar_test(y_true, pred1, pred2):
    """Compare two classifiers using McNemar's test."""
    c1 = (pred1 == y_true)
    c2 = (pred2 == y_true)
    b = np.sum(~c1 & c2)  # pred1 wrong, pred2 correct
    c = np.sum(c1 & ~c2)  # pred1 correct, pred2 wrong
    
    if b + c > 0:
        stat = (abs(b - c) - 1)**2 / (b + c)
        p = 1 - stats.chi2.cdf(stat, df=1)
    else:
        stat, p = 0, 1.0
    return b, c, stat, p

# CV comparison
b_cv, c_cv, stat_cv, p_cv = mcnemar_test(ovr.cv_true_labels, multi_cv_preds, ovr.cv_predictions)

# Task comparison  
b_task, c_task, stat_task, p_task = mcnemar_test(y_task, multi_task_preds, task_preds)

print("="*80)
print("McNEMAR'S TEST")
print("="*80)
print(f"\nCV: Multi wrong/OvR correct={b_cv}, Multi correct/OvR wrong={c_cv}")
print(f"    χ²={stat_cv:.2f}, p={p_cv:.4e}, Significant: {'Yes' if p_cv < 0.05 else 'No'}")
print(f"\nTask: Multi wrong/OvR correct={b_task}, Multi correct/OvR wrong={c_task}")
print(f"      χ²={stat_task:.2f}, p={p_task:.4e}, Significant: {'Yes' if p_task < 0.05 else 'No'}")

In [ ]:
# Error overlap analysis
multi_cv_err = set(np.where(multi_cv_preds != multi_cv_true)[0])
ovr_cv_err = set(np.where(ovr.cv_predictions != ovr.cv_true_labels)[0])

shared = len(multi_cv_err & ovr_cv_err)
only_multi = len(multi_cv_err - ovr_cv_err)
only_ovr = len(ovr_cv_err - multi_cv_err)

print("\nERROR OVERLAP (CV):")
print(f"  Both wrong: {shared:,}")
print(f"  Only Multinomial wrong: {only_multi:,}")
print(f"  Only OvR wrong: {only_ovr:,}")
print(f"  Overlap rate: {shared/(shared+only_multi+only_ovr):.1%}")

---
## 5. Error Taxonomy Analysis

In [ ]:
def analyze_errors(true_labels, pred_labels, region_info, name, condition):
    """Categorize errors by hemisphere and network boundaries."""
    err_mask = true_labels != pred_labels
    n_err = err_mask.sum()
    
    if n_err == 0:
        return {'Model': name, 'Condition': condition, 'Errors': 0, 'Rate': 0,
                'Within': 0, 'CrossNet': 0, 'CrossHemi': 0, 'Both': 0}
    
    lookup = region_info.set_index('region_idx')
    true_err, pred_err = true_labels[err_mask], pred_labels[err_mask]
    
    same_h = np.array([lookup.loc[t, 'hemisphere'] == lookup.loc[p, 'hemisphere'] 
                       for t, p in zip(true_err, pred_err)])
    same_n = np.array([lookup.loc[t, 'major_network'] == lookup.loc[p, 'major_network'] 
                       for t, p in zip(true_err, pred_err)])
    
    return {
        'Model': name, 'Condition': condition, 'Errors': n_err, 
        'Rate': n_err / len(true_labels),
        'Within': (same_h & same_n).sum(),
        'CrossNet': (same_h & ~same_n).sum(),
        'CrossHemi': (~same_h & same_n).sum(),
        'Both': (~same_h & ~same_n).sum()
    }

# Analyze all conditions
results = [
    analyze_errors(multi_cv_true, multi_cv_preds, region_info, 'Multinomial', 'Rest'),
    analyze_errors(multi_task_true, multi_task_preds, region_info, 'Multinomial', 'Task'),
    analyze_errors(ovr.cv_true_labels, ovr.cv_predictions, region_info, 'OvR', 'Rest'),
    analyze_errors(y_task, task_preds, region_info, 'OvR', 'Task')
]

err_df = pd.DataFrame(results)
for col in ['Within', 'CrossNet', 'CrossHemi', 'Both']:
    err_df[f'{col}_Pct'] = (err_df[col] / err_df['Errors'] * 100).round(1)

print("="*110)
print("ERROR TAXONOMY COMPARISON")
print("="*110)
print(f"\n{'Model':<12} {'Cond':<6} {'Errors':>7} {'Rate':>7} {'Within':>12} {'CrossNet':>12} {'CrossHemi':>12} {'Both':>12}")
print("-"*95)
for _, r in err_df.iterrows():
    print(f"{r['Model']:<12} {r['Condition']:<6} {r['Errors']:>7} {r['Rate']:>6.1%} "
          f"{r['Within']:>5} ({r['Within_Pct']:>4.1f}%) {r['CrossNet']:>5} ({r['CrossNet_Pct']:>4.1f}%) "
          f"{r['CrossHemi']:>5} ({r['CrossHemi_Pct']:>4.1f}%) {r['Both']:>5} ({r['Both_Pct']:>4.1f}%)")

In [ ]:
# Visualize error taxonomy
categories = ['Within_Pct', 'CrossNet_Pct', 'CrossHemi_Pct', 'Both_Pct']
labels = ['Within Network', 'Cross Network', 'Cross Hemisphere', 'Both Crossed']

fig = make_subplots(rows=1, cols=2, subplot_titles=['<b>Rest (CV)</b>', '<b>Task</b>'])
colors = {'Multinomial': '#2b8cbe', 'OvR': '#e6550d'}

for col_idx, cond in enumerate(['Rest', 'Task']):
    for model in ['Multinomial', 'OvR']:
        row = err_df[(err_df['Model'] == model) & (err_df['Condition'] == cond)].iloc[0]
        vals = [row[c] for c in categories]
        fig.add_trace(go.Bar(name=model if col_idx == 0 else None, x=labels, y=vals,
                             marker_color=colors[model], showlegend=(col_idx == 0)),
                      row=1, col=col_idx+1)

fig.update_layout(title='<b>Error Taxonomy: Multinomial vs OvR</b>', barmode='group',
                  template='simple_white', height=400, width=900)
fig.update_xaxes(tickangle=45)
fig.update_yaxes(title_text='% of Errors')
fig.show()

---
## 6. Confidence Calibration

In [ ]:
def analyze_confidence(probs, true_labels, pred_labels):
    """Analyze prediction confidence."""
    max_p = probs.max(axis=1)
    correct = true_labels == pred_labels
    
    return {
        'Correct_Conf': max_p[correct].mean(),
        'Incorrect_Conf': max_p[~correct].mean() if (~correct).sum() > 0 else 0,
        'Low_Conf_Correct': (max_p[correct] < 0.5).mean() if correct.sum() > 0 else 0,
        'High_Conf_Error': (max_p[~correct] > 0.5).mean() if (~correct).sum() > 0 else 0
    }

conf_multi_cv = analyze_confidence(multi_cv_probs, multi_cv_true, multi_cv_preds)
conf_multi_task = analyze_confidence(multi_task_probs, multi_task_true, multi_task_preds)
conf_ovr_cv = analyze_confidence(ovr.cv_probs_matrix, ovr.cv_true_labels, ovr.cv_predictions)
conf_ovr_task = analyze_confidence(task_probs, y_task, task_preds)

print("="*90)
print("CONFIDENCE CALIBRATION")
print("="*90)
print(f"\n{'Model':<12} {'Condition':<8} {'Correct Conf':>14} {'Incorrect Conf':>16} {'Low-Conf Correct':>18} {'High-Conf Error':>16}")
print("-"*88)
for name, cond, conf in [('Multinomial', 'Rest', conf_multi_cv), ('Multinomial', 'Task', conf_multi_task),
                          ('OvR', 'Rest', conf_ovr_cv), ('OvR', 'Task', conf_ovr_task)]:
    print(f"{name:<12} {cond:<8} {conf['Correct_Conf']:>13.1%} {conf['Incorrect_Conf']:>15.1%} "
          f"{conf['Low_Conf_Correct']:>17.1%} {conf['High_Conf_Error']:>15.1%}")

In [ ]:
# Confidence distribution plots
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    '<b>Multinomial - Rest</b>', '<b>Multinomial - Task</b>',
    '<b>OvR - Rest</b>', '<b>OvR - Task</b>'
])

configs = [
    (multi_cv_probs, multi_cv_true, multi_cv_preds, 1, 1),
    (multi_task_probs, multi_task_true, multi_task_preds, 1, 2),
    (ovr.cv_probs_matrix, ovr.cv_true_labels, ovr.cv_predictions, 2, 1),
    (task_probs, y_task, task_preds, 2, 2)
]

for probs, true_l, pred_l, row, col in configs:
    max_p = probs.max(axis=1)
    correct = true_l == pred_l
    
    fig.add_trace(go.Histogram(x=max_p[correct], name='Correct', opacity=0.7,
                               marker_color='#2b8cbe', nbinsx=40, histnorm='probability',
                               showlegend=(row==1 and col==1)), row=row, col=col)
    fig.add_trace(go.Histogram(x=max_p[~correct], name='Incorrect', opacity=0.7,
                               marker_color='#e6550d', nbinsx=40, histnorm='probability',
                               showlegend=(row==1 and col==1)), row=row, col=col)

fig.update_layout(title='<b>Confidence Distributions</b>', barmode='overlay',
                  template='simple_white', height=600, width=900)
fig.update_xaxes(range=[0, 1], title_text='Max Probability')
fig.update_yaxes(title_text='Proportion')
fig.show()

---
## 7. Network-Level Patterns

In [ ]:
def make_cm(true_l, pred_l, n):
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(true_l, pred_l): cm[t, p] += 1
    return cm

def aggregate_to_network(cm, region_info, networks):
    labels = region_info['major_network'].values
    df = pd.DataFrame(cm, index=labels, columns=labels)
    grouped = df.groupby(level=0).sum().T.groupby(level=0).sum().T
    return grouped.reindex(index=networks, columns=networks).fillna(0).values

# Create confusion matrices
ovr_cm_cv = make_cm(ovr.cv_true_labels, ovr.cv_predictions, N_REGIONS)
ovr_cm_task = make_cm(y_task, task_preds, N_REGIONS)

# Aggregate to network level
ovr_net_cv = aggregate_to_network(ovr_cm_cv, region_info, NETWORKS)
ovr_net_task = aggregate_to_network(ovr_cm_task, region_info, NETWORKS)
multi_net_cv = aggregate_to_network(multi_cm_cv, region_info, NETWORKS)
multi_net_task = aggregate_to_network(multi_cm_task, region_info, NETWORKS)

In [ ]:
# Network confusion heatmaps
fig = make_subplots(rows=2, cols=3, subplot_titles=[
    '<b>Multi: Rest</b>', '<b>Multi: Task</b>', '<b>Multi: Δ</b>',
    '<b>OvR: Rest</b>', '<b>OvR: Task</b>', '<b>OvR: Δ</b>'
], horizontal_spacing=0.08, vertical_spacing=0.12)

seq_cs = [[0, '#f7fbff'], [1, '#2b8cbe']]
div_cs = [[0, '#2b8cbe'], [0.5, '#fff'], [1, '#e6550d']]

mats = [(multi_net_cv, 1, 1, 'seq'), (multi_net_task, 1, 2, 'seq'), 
        (multi_net_task - multi_net_cv, 1, 3, 'div'),
        (ovr_net_cv, 2, 1, 'seq'), (ovr_net_task, 2, 2, 'seq'), 
        (ovr_net_task - ovr_net_cv, 2, 3, 'div')]

for mat, r, c, cs_type in mats:
    mask = np.eye(len(NETWORKS), dtype=bool)
    mat_m = np.where(mask, None, mat)
    if cs_type == 'seq':
        cs, zmin, zmax = seq_cs, 0, np.nanmax(mat_m[mat_m != None])
    else:
        cs = div_cs
        lim = np.nanmax(np.abs(mat_m[mat_m != None].astype(float)))
        zmin, zmax = -lim, lim
    
    fig.add_trace(go.Heatmap(z=mat_m, x=NETWORKS, y=NETWORKS, colorscale=cs,
                             zmin=zmin, zmax=zmax, text=mat_m, texttemplate='%{text:.0f}',
                             textfont=dict(size=7), showscale=False), row=r, col=c)

fig.update_layout(title='<b>Network-Level Confusion</b>', template='plotly_white',
                  height=700, width=1100)
fig.update_xaxes(tickangle=45, tickfont=dict(size=7))
fig.update_yaxes(autorange='reversed', tickfont=dict(size=7))
fig.show()

---
## 8. Per-Region Analysis

In [ ]:
def per_region_accuracy(true_l, pred_l, n):
    acc = {}
    for r in range(n):
        mask = true_l == r
        if mask.sum() > 0:
            acc[r] = (pred_l[mask] == r).mean()
    return acc

multi_cv_acc = per_region_accuracy(multi_cv_true, multi_cv_preds, N_REGIONS)
multi_task_acc = per_region_accuracy(multi_task_true, multi_task_preds, N_REGIONS)
ovr_cv_acc = per_region_accuracy(ovr.cv_true_labels, ovr.cv_predictions, N_REGIONS)
ovr_task_acc = per_region_accuracy(y_task, task_preds, N_REGIONS)

# Build comparison
region_comp = region_info[['region_idx', 'region_name', 'major_network', 'hemisphere']].copy()
region_comp['Multi_CV'] = region_comp['region_idx'].map(multi_cv_acc)
region_comp['Multi_Task'] = region_comp['region_idx'].map(multi_task_acc)
region_comp['OvR_CV'] = region_comp['region_idx'].map(ovr_cv_acc)
region_comp['OvR_Task'] = region_comp['region_idx'].map(ovr_task_acc)
region_comp['OvR_Improvement'] = region_comp['OvR_CV'] - region_comp['Multi_CV']
region_comp['OvR_Task_Drop'] = region_comp['OvR_CV'] - region_comp['OvR_Task']

In [ ]:
print("="*100)
print("REGIONS WITH LARGEST OvR IMPROVEMENT (CV)")
print("="*100)
top = region_comp.nlargest(10, 'OvR_Improvement')
print(f"\n{'Region':<40} {'Network':<18} {'Multi':>8} {'OvR':>8} {'Δ':>8}")
print("-"*85)
for _, r in top.iterrows():
    print(f"{r['region_name']:<40} {r['major_network']:<18} "
          f"{r['Multi_CV']:>7.1%} {r['OvR_CV']:>7.1%} {r['OvR_Improvement']:>+7.1%}")

print("\n" + "="*100)
print("REGIONS WITH LARGEST OvR DECLINE (CV)")
print("="*100)
bottom = region_comp.nsmallest(10, 'OvR_Improvement')
for _, r in bottom.iterrows():
    print(f"{r['region_name']:<40} {r['major_network']:<18} "
          f"{r['Multi_CV']:>7.1%} {r['OvR_CV']:>7.1%} {r['OvR_Improvement']:>+7.1%}")

In [ ]:
# Scatter plot: Multi vs OvR accuracy
fig = px.scatter(region_comp, x='Multi_CV', y='OvR_CV', color='major_network',
                 hover_data=['region_name'], opacity=0.7)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', 
                         line=dict(dash='dash', color='gray'), name='y=x'))
fig.update_layout(
    title='<b>Per-Region Accuracy: Multinomial vs OvR</b>',
    xaxis_title='Multinomial CV Accuracy', yaxis_title='OvR CV Accuracy',
    template='simple_white', height=500, width=700
)
fig.show()

---
## 9. Task-Induced Reorganization

In [ ]:
# Identify regions with largest task-induced accuracy drops
region_comp['Task_Sensitivity'] = region_comp['OvR_Task_Drop']

print("="*100)
print("REGIONS WITH LARGEST TASK-INDUCED ACCURACY DROP (Error-as-Signal)")
print("="*100)
print("\nThese regions show the most functional reorganization during task engagement:")
print(f"\n{'Region':<40} {'Network':<18} {'CV Acc':>8} {'Task Acc':>8} {'Drop':>8}")
print("-"*85)

sensitive = region_comp.nlargest(15, 'Task_Sensitivity')
for _, r in sensitive.iterrows():
    print(f"{r['region_name']:<40} {r['major_network']:<18} "
          f"{r['OvR_CV']:>7.1%} {r['OvR_Task']:>7.1%} {r['Task_Sensitivity']:>+7.1%}")

In [ ]:
# Network-level task sensitivity
net_sensitivity = region_comp.groupby('major_network').agg({
    'OvR_CV': 'mean', 'OvR_Task': 'mean', 'Task_Sensitivity': 'mean'
}).reset_index()
net_sensitivity = net_sensitivity.sort_values('Task_Sensitivity', ascending=False)

print("\n" + "="*80)
print("NETWORK-LEVEL TASK SENSITIVITY")
print("="*80)
print(f"\n{'Network':<28} {'CV Acc':>10} {'Task Acc':>10} {'Drop':>10}")
print("-"*60)
for _, r in net_sensitivity.iterrows():
    print(f"{r['major_network']:<28} {r['OvR_CV']:>9.1%} {r['OvR_Task']:>9.1%} {r['Task_Sensitivity']:>+9.1%}")

In [ ]:
# Volcano-style plot: Task sensitivity by network
fig = px.strip(region_comp, x='major_network', y='Task_Sensitivity', color='major_network',
               hover_data=['region_name'], stripmode='overlay')
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(
    title='<b>Task-Induced Accuracy Drop by Network</b><br><sup>Higher = more reorganization</sup>',
    xaxis_title='Network', yaxis_title='Accuracy Drop (CV - Task)',
    template='simple_white', height=500, showlegend=False
)
fig.update_xaxes(tickangle=45)
fig.show()

---
## 10. Conclusions

In [ ]:
# Summary statistics
print("="*100)
print("SUMMARY: OvR vs MULTINOMIAL COMPARISON")
print("="*100)

print(f"""
PERFORMANCE:
  • OvR CV Accuracy:       {ovr_cv_acc:.2%} (Multinomial: {multi_metrics['accuracy']:.2%}, Δ: {(ovr_cv_acc - multi_metrics['accuracy'])*100:+.2f}%)
  • OvR Task Accuracy:     {ovr_task_acc:.2%} (Multinomial: {multi_task['task_test_accuracy']:.2%}, Δ: {(ovr_task_acc - multi_task['task_test_accuracy'])*100:+.2f}%)
  • McNemar CV p-value:    {p_cv:.4e} ({'Significant' if p_cv < 0.05 else 'Not significant'})

ERROR PATTERNS:
  • OvR Within-Network errors:    {err_df[err_df['Model']=='OvR']['Within_Pct'].iloc[0]:.1f}% (Multinomial: {err_df[err_df['Model']=='Multinomial']['Within_Pct'].iloc[0]:.1f}%)
  • OvR Cross-Boundary errors:    {err_df[err_df['Model']=='OvR']['Both_Pct'].iloc[0]:.1f}% (Multinomial: {err_df[err_df['Model']=='Multinomial']['Both_Pct'].iloc[0]:.1f}%)

CONFIDENCE CALIBRATION:
  • OvR Correct Confidence:       {conf_ovr_cv['Correct_Conf']:.1%} (Multinomial: {conf_multi_cv['Correct_Conf']:.1%})
  • OvR Low-Conf Correct:         {conf_ovr_cv['Low_Conf_Correct']:.1%} (Multinomial: {conf_multi_cv['Low_Conf_Correct']:.1%})
  • OvR High-Conf Errors:         {conf_ovr_cv['High_Conf_Error']:.1%} (Multinomial: {conf_multi_cv['High_Conf_Error']:.1%})

TASK SENSITIVITY (Error-as-Signal):
  • Most sensitive network:       {net_sensitivity.iloc[0]['major_network']} (Drop: {net_sensitivity.iloc[0]['Task_Sensitivity']:.1%})
  • Least sensitive network:      {net_sensitivity.iloc[-1]['major_network']} (Drop: {net_sensitivity.iloc[-1]['Task_Sensitivity']:.1%})
""")

print("="*100)

In [ ]:
# Save results
results_summary = {
    'ovr_cv_accuracy': ovr_cv_acc,
    'ovr_task_accuracy': ovr_task_acc,
    'multi_cv_accuracy': multi_metrics['accuracy'],
    'multi_task_accuracy': multi_task['task_test_accuracy'],
    'mcnemar_cv_pvalue': p_cv,
    'mcnemar_task_pvalue': p_task
}
save_json(results_summary, OVR_DIR / 'full' / 'summary.json')
region_comp.to_csv(OVR_DIR / 'full' / 'region_comparison.csv', index=False)
cv_results_df.to_csv(OVR_DIR / 'full' / 'classifier_metrics.csv', index=False)

print("✓ Results saved to", OVR_DIR / 'full')

---

### Key Takeaways

1. **Accuracy**: Compare OvR vs. Multinomial to determine if decomposed classification improves performance.

2. **Error Patterns**: Examine if OvR produces more within-network errors (suggesting finer discriminability).

3. **Confidence**: Evaluate whether binary probabilities are better calibrated than softmax outputs.

4. **Task Sensitivity**: Use per-region accuracy drops to identify brain regions showing functional reorganization during task engagement—the core "error-as-signal" interpretation.

5. **Network Specificity**: Determine which functional networks are most affected by task demands (expected: Control, Salience/Ventral Attention for Gender Stroop task).